# Notebook 01 — Setup & Data Download

This notebook walks through:
1. Verifying the project environment
2. Loading the project configuration
3. Downloading NUTS-2 boundary shapefiles
4. Downloading ERA5 climate data (Recommended: **Earthmover**; Alternative: **CDS API**)

**Prerequisites:** Run `uv sync` in the project root before starting.

## 1. Verify Environment & Imports

In [ ]:
import sys
import os

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

In [ ]:
# Test that key packages are available
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import arraylake

print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"xarray:     {xr.__version__}")
print(f"geopandas:  {gpd.__version__}")
print(f"arraylake:  {arraylake.__version__}")
print("\n✅ All key packages imported successfully")

## 2. Load Project Configuration

In [ ]:
from src.utils.config import load_config, get_path, get_region_mapping

cfg = load_config()

print(f"Study period: {cfg['study']['start_year']}–{cfg['study']['end_year']}")
print(f"Summer months: {cfg['study']['summer_months']}")
print(f"Configured climate source: {cfg['temperature']['primary_dataset']}")

## 3. Download NUTS-2 Boundaries

In [ ]:
from src.data.nuts2_boundaries import setup_boundaries

# Download boundaries (skips if already downloaded)
italy_nuts2 = setup_boundaries(cfg)

print(f"\nLoaded {len(italy_nuts2)} Italian NUTS-2 regions")

# Quick map
fig, ax = plt.subplots(1, 1, figsize=(6, 8))
italy_nuts2.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
ax.set_title('Italian NUTS-2 Regions', fontweight='bold')
ax.set_axis_off()
plt.show()

## 4. Download ERA5 Climate Data

### Recommended: Earthmover (AWS Zarr)
Streams regional subsets from AWS. No account registration required for public data.
**⏱ Time:** ~5 minutes for 11 years.

### Alternative: Copernicus CDS (API)
Requires registration and API key. 
**⏱ Time:** ~4–6 hours.

In [ ]:
# Download via Earthmover (Recommended)
from src.data.download_earthmover import download_all_earthmover

# Set primary_dataset to 'earthmover' in config.yaml to use this by default
if cfg['temperature']['primary_dataset'] == 'earthmover':
    print("Starting Earthmover download...")
    # download_all_earthmover(cfg)  # Uncomment to run
else:
    print("Primary source is not Earthmover. Skipping automated cellular download.")

In [ ]:
# Verify downloaded files
climate_dir = get_path(cfg, 'raw_data') / 'climate'
files = sorted(list(climate_dir.glob('*.nc')))
print(f"Found {len(files)} climate files in {climate_dir}:")
for f in files[:5]:
    print(f"  {f.name}")
if len(files) > 5:
    print("  ...")

➡️ **Next:** `02_process_data.ipynb` — Process raw data into analysis-ready format.